# Port reference data preparation

## Purpose

This notebook prepares the external port-reference datasets used by the EFFKG
Wikibase import pipeline and performs a complementary overlap analysis between
the World Port Index (WPI) and the CNSP ERS port list.

The notebook has two operational purposes:

1. prepare filtered WPI and CNSP port files used during vessel ingestion to
   resolve or create vessel registration ports;
2. analyse cross-source overlap, matching behaviour and cardinality issues
   between both port-reference resources.

## Pipeline

1. Filter the World Port Index to the configured European geographic scope.
2. Normalize WPI UN/LOCODE values and geographic coordinates.
3. Filter the CNSP ERS port list to European Union countries and associated
   overseas territories.
4. Normalize CNSP identifiers, coordinates and FAO fishing-area values.
5. Export the prepared WPI and CNSP files consumed by the Wikibase import
   notebook.
6. Compare both resources using exact UN/LOCODE matching.
7. Compare remaining records using exact equality of normalized port names.
8. Generate a consolidated overlap table for analysis.
9. Export audit files describing normalized-name matches and match-cardinality
   issues.

## Operational outputs

The following files are consumed by `import_wikibase.ipynb`:

- `wpi_european_ports.csv`
- `cnsp_ports_ue_filtered.csv`

They are used when vessel registration places cannot be resolved against
existing Wikibase port entities.

## Analytical outputs

The following files support source assessment and validation but are not
imported directly into Wikibase:

- `port_reference_overlap.csv`
- `ports_textual_matches.csv`
- `ports_cardinality_issues.csv`
- `ports_cardinality_issue_rows.csv`

# Setup

In [ ]:
# =============================================================================
# SETUP AND CONFIGURATION
# =============================================================================

from pathlib import Path
from typing import Optional

import re
import pandas as pd


def find_repository_root(start: Optional[Path] = None) -> Path:
    """
    Locate the EFFKG repository root from the current working directory.

    The repository root is identified by the presence of the main project
    directories used by the published pipeline.
    """
    current = (start or Path.cwd()).resolve()

    required_directories = {
        "code",
        "dataset",
        "schema",
        "source_data",
        "data_model",
        "validation",
    }

    for candidate in [current] + list(current.parents):
        try:
            existing_directories = {
                path.name
                for path in candidate.iterdir()
                if path.is_dir()
            }
        except PermissionError:
            continue

        if required_directories.issubset(existing_directories):
            return candidate

    raise RuntimeError(
        "EFFKG repository root could not be located. "
        "Run this notebook from within a cloned EFFKG repository."
    )


REPOSITORY_ROOT = find_repository_root()


# -----------------------------------------------------------------------------
# Input files
# -----------------------------------------------------------------------------

# World Port Index input.
# This file must contain the WPI data used for the current release.
WPI_INPUT_FILE = (
    REPOSITORY_ROOT
    / "source_data"
    / "wpi_ports.csv"
)

# CNSP ERS port list input.
CNSP_INPUT_FILE = (
    REPOSITORY_ROOT
    / "source_data"
    / "cnsp_ports.csv"
)


# -----------------------------------------------------------------------------
# Output directories
# -----------------------------------------------------------------------------

PORT_OUTPUT_DIRECTORY = (
    REPOSITORY_ROOT
    / "source_data"
    / "processed"
    / "ports"
)

PORT_VALIDATION_DIRECTORY = (
    REPOSITORY_ROOT
    / "validation"
    / "ports"
)

PORT_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

PORT_VALIDATION_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# -----------------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------------

WPI_FILTERED_FILE = (
    PORT_OUTPUT_DIRECTORY
    / "wpi_european_ports.csv"
)

CNSP_FILTERED_FILE = (
    PORT_OUTPUT_DIRECTORY
    / "cnsp_ports_ue_filtered.csv"
)

CONSOLIDATED_OUTPUT_FILE = (
    PORT_OUTPUT_DIRECTORY
    / "port_reference_overlap.csv"
)

AUDIT_OUTPUT_PREFIX = (
    PORT_VALIDATION_DIRECTORY
    / "ports"
)


# -----------------------------------------------------------------------------
# Source metadata
# -----------------------------------------------------------------------------

CNSP_SOURCE_URL = (
    "https://www.data.gouv.fr/api/1/datasets/r/"
    "60fe965d-5888-493b-9321-24bc3b1f84db"
)

WPI_SOURCE = "WPI"


# -----------------------------------------------------------------------------
# Validate required inputs
# -----------------------------------------------------------------------------

required_input_files = [
    WPI_INPUT_FILE,
    CNSP_INPUT_FILE,
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.is_file()
]

if missing_input_files:
    missing_paths = "\n".join(
        "  - {}".format(path)
        for path in missing_input_files
    )

    raise FileNotFoundError(
        "One or more required input files were not found:\n"
        "{}\n\n"
        "Place the required source files in source_data/ "
        "before running the notebook. "
        "See the input documentation for the expected filenames "
        "and column structure.".format(missing_paths)
    )


print("EFFKG port overlap analysis")
print("---------------------------")
print("Repository root: {}".format(REPOSITORY_ROOT))
print("WPI input: {}".format(WPI_INPUT_FILE))
print("CNSP input: {}".format(CNSP_INPUT_FILE))
print("Processed outputs: {}".format(PORT_OUTPUT_DIRECTORY))
print("Validation outputs: {}".format(PORT_VALIDATION_DIRECTORY))

EFFKG port overlap analysis
---------------------------
Repository root: C:\Users\profesor\Documents\EFFKG
WPI input: C:\Users\profesor\Documents\EFFKG\source_data\wpi_ports2.csv
CNSP input: C:\Users\profesor\Documents\EFFKG\source_data\cnsp_ports.csv
Processed outputs: C:\Users\profesor\Documents\EFFKG\source_data\processed\ports
Validation outputs: C:\Users\profesor\Documents\EFFKG\validation\ports


In [ ]:
# Country names as represented in the WPI file.
WPI_EUROPEAN_COUNTRIES = {
    'France', 'Spain', 'Italy', 'Portugal', 'Germany', 'Belgium', 'Netherlands',
    'United Kingdom', 'Ireland', 'Denmark', 'Sweden', 'Norway', 'Finland',
    'Poland', 'Lithuania', 'Latvia', 'Estonia', 'Iceland', 'Malta', 'Cyprus',
    'Croatia', 'Slovenia', 'Greece', 'Monaco', 'Albania', 'Montenegro',
    'Bosnia and Herzegovina', 'Romania', 'Bulgaria', 'Ukraine', 'Russia',
    'Turkey', 'Georgia'
}

# EU countries as ISO2 codes in the CNSP file.
EU_COUNTRIES = {
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR",
    "DE", "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL",
    "PL", "PT", "RO", "SK", "SI", "ES", "SE"
}

# Overseas territories linked to EU countries.
EU_OVERSEAS_TERRITORIES = {
    "GF",  # French Guiana
    "GP",  # Guadeloupe
    "MQ",  # Martinique
    "RE",  # Réunion
    "YT",  # Mayotte
    "PM",  # Saint Pierre and Miquelon
    "BL",  # Saint Barthélemy
    "MF",  # Saint Martin, French part
    "NC",  # New Caledonia
    "PF",  # French Polynesia
    "WF",  # Wallis and Futuna
    "TF",  # French Southern Territories
    "AW",  # Aruba
    "CW",  # Curaçao
    "SX",  # Sint Maarten
    "BQ",  # Caribbean Netherlands
    "GL",  # Greenland
    "FO"   # Faroe Islands
}

VALID_CNSP_CODES = EU_COUNTRIES | EU_OVERSEAS_TERRITORIES

INVALID_VALUES = {
    "",
    "PORT_NAME",
    "LOCODE",
    "MAIN PORT NAME",
    "UN/LOCODE",
    "PUBLICATION LINK",
    "COUNTRY CODE",
    "REGION NAME",
    "POSITION",
    "LATITUDE",
    "LONGITUDE",
    "SOURCE",
    "FAO_AREAS",
}

# Helpers

In [ ]:
def as_clean_string(value):
    """Return a stripped string, or an empty string for missing values."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def is_invalid_value(value):
    """Check whether a value is empty or looks like a header accidentally parsed as data."""
    text = as_clean_string(value).upper()
    return text in INVALID_VALUES


def clean_port_name(value):
    """Clean a port name and discard invalid/header-like values."""
    text = as_clean_string(value)
    if is_invalid_value(text):
        return ""
    return text


def clean_locode(value):
    """Clean a UN/LOCODE value and discard invalid/header-like values."""
    text = as_clean_string(value).upper()
    if is_invalid_value(text):
        return ""
    return text


def normalize_text(value):
    """Normalize text for matching purposes while preserving display values elsewhere."""
    text = clean_port_name(value).upper()
    if not text:
        return ""
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_coordinates(lat, lon):
    """Build a lat,lon,precision coordinate string from latitude and longitude columns."""
    if pd.isna(lat) or pd.isna(lon):
        return ""
    lat = str(lat).strip()
    lon = str(lon).strip()
    if not lat or not lon:
        return ""
    return f"{lat},{lon},0.000001"


def parse_wpi_coordinate(value, coordinate_type):
    """
    Parse a coordinate from the official WPI export.

    Some spreadsheet exports represent coordinates using values such as:

        4,05333E+16
        -7,425E+16

    although the intended coordinates are:

        40.5333
        -74.25

    The value is converted to a float and repeatedly divided by 10 until it
    falls within the valid geographic range.

    Parameters
    ----------
    value:
        Raw coordinate value.

    coordinate_type:
        Either "latitude" or "longitude".

    Returns
    -------
    float or None
        Normalised coordinate, or None when the value is missing or invalid.
    """
    if pd.isna(value):
        return None

    text = str(value).strip()

    if not text:
        return None

    # Support decimal commas and remove spaces occasionally introduced by
    # spreadsheet exports.
    text = text.replace(" ", "").replace(",", ".")

    try:
        coordinate = float(text)
    except (TypeError, ValueError):
        return None

    if coordinate_type == "latitude":
        maximum_absolute_value = 90.0
    elif coordinate_type == "longitude":
        maximum_absolute_value = 180.0
    else:
        raise ValueError(
            "coordinate_type must be 'latitude' or 'longitude'."
        )

    # Repair scientific-notation values whose magnitude was expanded during
    # spreadsheet export.
    while abs(coordinate) > maximum_absolute_value:
        coordinate /= 10.0

    if abs(coordinate) > maximum_absolute_value:
        return None

    return coordinate


def wpi_coordinates_to_position(latitude, longitude):
    """
    Build the Wikibase coordinate representation expected downstream:

        latitude,longitude,precision
    """
    latitude = parse_wpi_coordinate(
        latitude,
        "latitude"
    )

    longitude = parse_wpi_coordinate(
        longitude,
        "longitude"
    )

    if latitude is None or longitude is None:
        return ""

    return "{:.6f},{:.6f},0.000001".format(
        latitude,
        longitude
    )


def merge_sources(*values):
    """Merge source markers into a comma-separated unique list."""
    items = []
    for value in values:
        if pd.isna(value) or value is None:
            continue
        for part in str(value).split(","):
            part = part.strip()
            if part and part not in items:
                items.append(part)
    return ",".join(items)


def first_not_empty(*values):
    """Return the first non-empty value from the provided candidates."""
    for value in values:
        if pd.notna(value) and str(value).strip() != "":
            return value
    return ""


def build_country_region(country, region):
    """Build a compact country-region string for the consolidated port output."""
    country = as_clean_string(country)
    region = as_clean_string(region)

    if is_invalid_value(country):
        country = ""
    if is_invalid_value(region):
        region = ""

    parts = [x for x in [country, region] if x]
    return " - ".join(parts)


# =========================
# FAO area helpers
# =========================

def parse_fao_areas(value):
    """
    Parse FAO area lists while preserving comma-suffixed fragments.

    Examples:
        {34.3.4,34.4.1} -> ['34.3.4', '34.4.1']
        {27.3.b,c,27.3.c.33} -> ['27.3.b,c', '27.3.c.33']

    Rule:
        - split by comma
        - if a token is only letters, attach it to the previous token
    """
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text or text == "{}":
        return []

    if text.startswith("{") and text.endswith("}"):
        text = text[1:-1].strip()

    if not text:
        return []

    raw_parts = [p.strip() for p in text.split(",") if p.strip()]
    if not raw_parts:
        return []

    merged = []
    for part in raw_parts:
        if re.fullmatch(r"[A-Za-z]+", part) and merged:
            merged[-1] = f"{merged[-1]},{part}"
        else:
            merged.append(part)

    # Remove duplicates while preserving order.
    seen = set()
    result = []
    for item in merged:
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result


def format_fao_areas(value):
    """Format FAO areas as repeated braced values expected by the downstream import."""
    areas = parse_fao_areas(value)
    if not areas:
        return ""
    return ",".join(f"{{{area}}}" for area in areas)

# Step 1: Prepare European World Port Index records

In [ ]:
def filter_wpi_ports():
    """
    Filter the official World Port Index to the configured European scope,
    normalize UN/LOCODE values and convert coordinates to the format expected
    by the Wikibase import pipeline.
    """
    df = pd.read_csv(
        WPI_INPUT_FILE,
        dtype=str
    )

    required_columns = {
        "Country Code",
        "Main Port Name",
        "UN/LOCODE",
        "Latitude",
        "Longitude",
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            "The World Port Index input is missing required columns: {}"
            .format(", ".join(sorted(missing_columns)))
        )

    filtered = df[
        df["Country Code"]
        .fillna("")
        .astype(str)
        .str.strip()
        .isin(WPI_EUROPEAN_COUNTRIES)
    ].copy()

    # Remove internal spaces from UN/LOCODE values.
    filtered["UN/LOCODE"] = (
        filtered["UN/LOCODE"]
        .fillna("")
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
        .str.upper()
    )

    # Keep the cleaned numeric coordinate values in their original columns.
    filtered["Latitude"] = filtered["Latitude"].apply(
        lambda value: parse_wpi_coordinate(
            value,
            "latitude"
        )
    )

    filtered["Longitude"] = filtered["Longitude"].apply(
        lambda value: parse_wpi_coordinate(
            value,
            "longitude"
        )
    )

    # Build the coordinate representation expected by the downstream code.
    filtered["Position"] = filtered.apply(
        lambda row: wpi_coordinates_to_position(
            row.get("Latitude"),
            row.get("Longitude")
        ),
        axis=1
    )

    filtered.to_csv(
        WPI_FILTERED_FILE,
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    valid_coordinates = (
        filtered["Position"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    print("WPI file successfully generated")
    print("WPI exported rows: {}".format(len(filtered)))
    print(
        "WPI rows with valid coordinates: {}".format(
            valid_coordinates
        )
    )
    print(
        "WPI rows without valid coordinates: {}".format(
            len(filtered) - valid_coordinates
        )
    )

# Step 2: Prepare European CNSP ERS port records

In [ ]:
def filter_cnsp_ports():
    """
    Filter the CNSP ERS port list to EU countries and associated overseas
    territories, normalize coordinates and attach the source URL used for
    statement-level references.
    """
    df = pd.read_csv(CNSP_INPUT_FILE)

    # Normalize country codes before filtering.
    df["country_code_iso2"] = df["country_code_iso2"].astype(str).str.upper().str.strip()

    # Keep only EU countries and associated overseas territories.
    df = df[df["country_code_iso2"].isin(VALID_CNSP_CODES)].copy()

    # Create coordinates column with format: lat,lon,precision.
    df["coordinates"] = df.apply(
        lambda row: make_coordinates(row.get("latitude"), row.get("longitude")),
        axis=1
    )

    # Add source column for downstream references.
    df["source"] = CNSP_SOURCE_URL

    df.to_csv(CNSP_FILTERED_FILE, sep=";", index=False, encoding="utf-8")

    print(f"CNSP file generated: {CNSP_FILTERED_FILE}")
    print(f"CNSP resulting rows: {len(df)}")

# Step 3: Compare and consolidate prepared port-reference files

In [ ]:
def load_prepared_ports():
    """
    Load the prepared WPI and CNSP files generated by the operational
    preparation stages.
    """
    cnsp = pd.read_csv(CNSP_FILTERED_FILE, sep=";")
    wpi = pd.read_csv(WPI_FILTERED_FILE, sep=";")
    return cnsp, wpi


def prepare_cnsp_for_matching(cnsp):
    """Rename and normalize CNSP columns before matching."""
    cnsp = cnsp.rename(columns={
        "port_name": "cnsp_port_name",
        "locode": "cnsp_locode",
        "region": "cnsp_region",
        "country_code_iso2": "cnsp_country",
        "fao_areas": "cnsp_fao_areas",
        "coordinates": "cnsp_coordinates",
        "source": "cnsp_source"
    })

    cnsp["cnsp_port_name"] = cnsp["cnsp_port_name"].apply(clean_port_name)
    cnsp["cnsp_locode"] = cnsp["cnsp_locode"].apply(clean_locode)
    cnsp["cnsp_locode_clean"] = cnsp["cnsp_locode"]
    cnsp["cnsp_name_clean"] = cnsp["cnsp_port_name"].apply(normalize_text)
    cnsp["cnsp_fao_areas"] = cnsp["cnsp_fao_areas"].apply(format_fao_areas)

    if "latitude" in cnsp.columns and "longitude" in cnsp.columns:
        missing = cnsp["cnsp_coordinates"].isna() | (cnsp["cnsp_coordinates"].astype(str).str.strip() == "")
        cnsp.loc[missing, "cnsp_coordinates"] = cnsp.loc[missing].apply(
            lambda row: make_coordinates(row.get("latitude"), row.get("longitude")),
            axis=1
        )

    cnsp["cnsp_country_region"] = cnsp.apply(
        lambda row: build_country_region(row.get("cnsp_country"), row.get("cnsp_region")),
        axis=1
    )

    # Keep rows with at least one usable identifying value.
    cnsp = cnsp[
        (cnsp["cnsp_port_name"].astype(str).str.strip() != "") |
        (cnsp["cnsp_locode_clean"].astype(str).str.strip() != "")
    ].copy()

    return cnsp


def prepare_wpi_for_matching(wpi):
    """Rename and normalize WPI columns before matching."""
    wpi = wpi.rename(columns={
        "Main Port Name": "wpi_port_name",
        "UN/LOCODE": "wpi_locode",
        "Country Code": "wpi_country",
        "Region Name": "wpi_region",
        "Latitude": "wpi_latitude",
        "Longitude": "wpi_longitude",
        "Position": "wpi_coordinates"
    })

    wpi["wpi_port_name"] = wpi["wpi_port_name"].apply(clean_port_name)
    wpi["wpi_locode"] = wpi["wpi_locode"].apply(clean_locode)
    wpi["wpi_locode_clean"] = wpi["wpi_locode"]
    wpi["wpi_name_clean"] = wpi["wpi_port_name"].apply(normalize_text)
    wpi["wpi_source"] = WPI_SOURCE

    missing = wpi["wpi_coordinates"].isna() | (wpi["wpi_coordinates"].astype(str).str.strip() == "")
    wpi.loc[missing, "wpi_coordinates"] = wpi.loc[missing].apply(
        lambda row: make_coordinates(row.get("wpi_latitude"), row.get("wpi_longitude")),
        axis=1
    )

    wpi["wpi_country_region"] = wpi.apply(
        lambda row: build_country_region(row.get("wpi_country"), row.get("wpi_region")),
        axis=1
    )

    # Keep rows with at least one usable identifying value.
    wpi = wpi[
        (wpi["wpi_port_name"].astype(str).str.strip() != "") |
        (wpi["wpi_locode_clean"].astype(str).str.strip() != "")
    ].copy()

    return wpi


def match_ports(cnsp, wpi):
    """
    Compare prepared CNSP and WPI records in two deterministic stages.

    Stage 1:
        Exact equality of cleaned UN/LOCODE values.

    Stage 2:
        Exact equality of normalized port names for records not matched during
        the UN/LOCODE stage.

    This function does not use fuzzy matching, geographic distance or manual
    adjudication.
    """
    # First pass: match by LOCODE.
    cnsp_by_locode = cnsp[cnsp["cnsp_locode_clean"] != ""].copy()
    wpi_by_locode = wpi[wpi["wpi_locode_clean"] != ""].copy()

    matched_locode = pd.merge(
        cnsp_by_locode,
        wpi_by_locode,
        left_on="cnsp_locode_clean",
        right_on="wpi_locode_clean",
        how="inner"
    )

    matched_cnsp_locodes = set(matched_locode["cnsp_locode_clean"].dropna())
    matched_wpi_locodes = set(matched_locode["wpi_locode_clean"].dropna())

    cnsp_left = cnsp[~cnsp["cnsp_locode_clean"].isin(matched_cnsp_locodes)].copy()
    wpi_left = wpi[~wpi["wpi_locode_clean"].isin(matched_wpi_locodes)].copy()

    # Second pass: match remaining rows by normalized name.
    cnsp_by_name = cnsp_left[cnsp_left["cnsp_name_clean"] != ""].copy()
    wpi_by_name = wpi_left[wpi_left["wpi_name_clean"] != ""].copy()

    matched_name = pd.merge(
        cnsp_by_name,
        wpi_by_name,
        left_on="cnsp_name_clean",
        right_on="wpi_name_clean",
        how="inner"
    )

    matched_cnsp_names = set(matched_name["cnsp_name_clean"].dropna())
    matched_wpi_names = set(matched_name["wpi_name_clean"].dropna())

    cnsp_only = cnsp_left[~cnsp_left["cnsp_name_clean"].isin(matched_cnsp_names)].copy()
    wpi_only = wpi_left[~wpi_left["wpi_name_clean"].isin(matched_wpi_names)].copy()

    return matched_locode, matched_name, cnsp_only, wpi_only

def audit_port_overlap(
    cnsp,
    wpi,
    output_prefix="port_overlap"
):
    """
    Audit cross-source matching results without modifying the operational WPI
    or CNSP prepared files or the consolidated overlap output.

    Generates:
      1. <prefix>_textual_matches.csv
      2. <prefix>_cardinality_issues.csv
      3. <prefix>_cardinality_issue_rows.csv
    """

    # Work only on copies.
    cnsp_audit = cnsp.copy()
    wpi_audit = wpi.copy()

    # Temporary source-row identifiers for cardinality analysis.
    cnsp_audit["_cnsp_row_id"] = range(len(cnsp_audit))
    wpi_audit["_wpi_row_id"] = range(len(wpi_audit))

    (
        matched_locode,
        matched_name,
        cnsp_only,
        wpi_only
    ) = match_ports(cnsp_audit, wpi_audit)

    # ============================================================
    # 1. EXPORT THE PORTS MATCHED BY NORMALIZED NAME
    # ============================================================

    textual_columns = [
        "_cnsp_row_id",
        "_wpi_row_id",
        "cnsp_port_name",
        "wpi_port_name",
        "cnsp_name_clean",
        "cnsp_locode",
        "wpi_locode",
        "cnsp_country",
        "wpi_country",
        "cnsp_region",
        "wpi_region",
        "cnsp_coordinates",
        "wpi_coordinates"
    ]

    textual_columns = [
        column
        for column in textual_columns
        if column in matched_name.columns
    ]

    textual_matches = matched_name[textual_columns].copy()

    # Make the reason for the match explicit.
    textual_matches.insert(
        0,
        "matching_method",
        "normalized_port_name"
    )

    textual_matches.to_csv(
        f"{output_prefix}_textual_matches.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nPORTS MATCHED BY NORMALIZED NAME")
    print("--------------------------------")
    print(textual_matches.to_string(index=False))

    # ============================================================
    # 2. IDENTIFY CARDINALITY PROBLEMS
    # ============================================================

    cardinality_tables = []

    def summarize_cardinality(
        matched_df,
        key_column,
        method
    ):
        if matched_df.empty:
            return pd.DataFrame()

        summary = (
            matched_df
            .groupby(key_column, dropna=False)
            .agg(
                cnsp_source_rows=("_cnsp_row_id", "nunique"),
                wpi_source_rows=("_wpi_row_id", "nunique"),
                generated_merge_rows=("_cnsp_row_id", "size")
            )
            .reset_index()
        )

        summary.insert(0, "matching_method", method)
        summary = summary.rename(
            columns={key_column: "matching_key"}
        )

        # Under a strict one-to-one match:
        # one CNSP row + one WPI row -> one output row.
        #
        # This value measures the excess generated by duplicated keys:
        #
        # 2 * merge rows - distinct CNSP rows - distinct WPI rows
        #
        # For a 1-to-2 match:
        # 2*2 - 1 - 2 = 1 additional row.
        summary["additional_rows_from_cardinality"] = (
            2 * summary["generated_merge_rows"]
            - summary["cnsp_source_rows"]
            - summary["wpi_source_rows"]
        )

        return summary[
            summary["additional_rows_from_cardinality"] > 0
        ].copy()

    locode_issues = summarize_cardinality(
        matched_locode,
        "cnsp_locode_clean",
        "exact_locode"
    )

    name_issues = summarize_cardinality(
        matched_name,
        "cnsp_name_clean",
        "normalized_port_name"
    )

    if not locode_issues.empty:
        cardinality_tables.append(locode_issues)

    if not name_issues.empty:
        cardinality_tables.append(name_issues)

    if cardinality_tables:
        cardinality_issues = pd.concat(
            cardinality_tables,
            ignore_index=True
        )
    else:
        cardinality_issues = pd.DataFrame(
            columns=[
                "matching_method",
                "matching_key",
                "cnsp_source_rows",
                "wpi_source_rows",
                "generated_merge_rows",
                "additional_rows_from_cardinality"
            ]
        )

    cardinality_issues.to_csv(
        f"{output_prefix}_cardinality_issues.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nPORT MATCH CARDINALITY ISSUES")
    print("-----------------------------")

    if cardinality_issues.empty:
        print("No one-to-many or many-to-many matches detected.")
    else:
        print(cardinality_issues.to_string(index=False))

    total_additional_rows = int(
        cardinality_issues[
            "additional_rows_from_cardinality"
        ].sum()
    )

    print(
        "\nTotal additional rows generated by match cardinality:",
        total_additional_rows
    )

    # ============================================================
    # 3. EXPORT THE ACTUAL SOURCE ROWS INVOLVED
    # ============================================================

    problematic_rows = []

    for _, issue in cardinality_issues.iterrows():
        method = issue["matching_method"]
        key = issue["matching_key"]

        if method == "exact_locode":
            affected = matched_locode[
                matched_locode["cnsp_locode_clean"] == key
            ].copy()
        else:
            affected = matched_name[
                matched_name["cnsp_name_clean"] == key
            ].copy()

        affected.insert(0, "audit_matching_method", method)
        affected.insert(1, "audit_matching_key", key)

        problematic_rows.append(affected)

    if problematic_rows:
        cardinality_issue_rows = pd.concat(
            problematic_rows,
            ignore_index=True
        )

        useful_columns = [
            "audit_matching_method",
            "audit_matching_key",
            "_cnsp_row_id",
            "_wpi_row_id",
            "cnsp_port_name",
            "wpi_port_name",
            "cnsp_locode",
            "wpi_locode",
            "cnsp_country",
            "wpi_country",
            "cnsp_region",
            "wpi_region",
            "cnsp_coordinates",
            "wpi_coordinates"
        ]

        useful_columns = [
            column
            for column in useful_columns
            if column in cardinality_issue_rows.columns
        ]

        cardinality_issue_rows = cardinality_issue_rows[
            useful_columns
        ]

    else:
        cardinality_issue_rows = pd.DataFrame()

    cardinality_issue_rows.to_csv(
        f"{output_prefix}_cardinality_issue_rows.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nROWS INVOLVED IN CARDINALITY ISSUES")
    print("-----------------------------------")

    if cardinality_issue_rows.empty:
        print("No problematic rows.")
    else:
        print(cardinality_issue_rows.to_string(index=False))

    # ============================================================
    # 4. GENERAL CONSISTENCY CHECK
    # ============================================================

    total_matches = len(matched_locode) + len(matched_name)

    distinct_cnsp_matched = len(
        set(matched_locode["_cnsp_row_id"])
        | set(matched_name["_cnsp_row_id"])
    )

    distinct_wpi_matched = len(
        set(matched_locode["_wpi_row_id"])
        | set(matched_name["_wpi_row_id"])
    )

    calculated_difference = (
        2 * total_matches
        - distinct_cnsp_matched
        - distinct_wpi_matched
    )

    print("\nCARDINALITY CONSISTENCY CHECK")
    print("-----------------------------")
    print(f"merge_rows_generated: {total_matches}")
    print(f"distinct_matched_cnsp_rows: {distinct_cnsp_matched}")
    print(f"distinct_matched_wpi_rows: {distinct_wpi_matched}")
    print(
        "additional_rows_explained_by_cardinality:",
        calculated_difference
    )

    return {
        "textual_matches": textual_matches,
        "cardinality_issues": cardinality_issues,
        "cardinality_issue_rows": cardinality_issue_rows
    }

def print_port_overlap_statistics(
    cnsp,
    wpi,
    matched_locode,
    matched_name,
    cnsp_only,
    wpi_only,
    final_df
):
    """
    Print overlap statistics without modifying the
    consolidated dataframe or the output CSV.
    """

    locode_matches = len(matched_locode)
    name_matches = len(matched_name)
    total_matches = locode_matches + name_matches

    locode_pct = (
        100 * locode_matches / total_matches
        if total_matches else 0.0
    )

    name_pct = (
        100 * name_matches / total_matches
        if total_matches else 0.0
    )

    print("\nPORT OVERLAP STATISTICS")
    print("------------------------------")
    print(f"cnsp_source_records: {len(cnsp)}")
    print(f"wpi_source_records: {len(wpi)}")
    print(f"matched_by_locode: {locode_matches}")
    print(f"matched_by_normalized_name: {name_matches}")
    print(f"total_cross_source_matches: {total_matches}")
    print(f"locode_matches_pct: {locode_pct:.2f}")
    print(f"normalized_name_matches_pct: {name_pct:.2f}")
    print(f"unmatched_cnsp_records: {len(cnsp_only)}")
    print(f"unmatched_wpi_records: {len(wpi_only)}")
    print(f"final_consolidated_rows: {len(final_df)}")


def build_output(df):
    """Build final-schema rows from matched CNSP/WPI records."""
    rows = []

    for _, row in df.iterrows():
        port_name = first_not_empty(row.get("wpi_port_name"), row.get("cnsp_port_name"))
        locode = first_not_empty(row.get("wpi_locode"), row.get("cnsp_locode"))
        coordinates = first_not_empty(row.get("wpi_coordinates"), row.get("cnsp_coordinates"))
        country_region = first_not_empty(row.get("wpi_country_region"), row.get("cnsp_country_region"))
        fao_areas = first_not_empty(row.get("cnsp_fao_areas"))
        sources = merge_sources(row.get("cnsp_source"), row.get("wpi_source"))

        port_name = clean_port_name(port_name)
        locode = clean_locode(locode)

        if port_name or locode:
            rows.append({
                "sources": sources,
                "port_name": port_name,
                "coordinates": coordinates,
                "locode": locode,
                "country_region": country_region,
                "fao_areas": fao_areas
            })

    return pd.DataFrame(rows)


def build_cnsp_only_output(cnsp_only):
    """Build final-schema rows for unmatched CNSP records."""
    return pd.DataFrame([{
        "sources": row.get("cnsp_source", ""),
        "port_name": clean_port_name(row.get("cnsp_port_name", "")),
        "coordinates": row.get("cnsp_coordinates", ""),
        "locode": clean_locode(row.get("cnsp_locode", "")),
        "country_region": row.get("cnsp_country_region", ""),
        "fao_areas": row.get("cnsp_fao_areas", "")
    } for _, row in cnsp_only.iterrows()
      if clean_port_name(row.get("cnsp_port_name", "")) or clean_locode(row.get("cnsp_locode", ""))])


def build_wpi_only_output(wpi_only):
    """Build final-schema rows for unmatched WPI records."""
    return pd.DataFrame([{
        "sources": row.get("wpi_source", ""),
        "port_name": clean_port_name(row.get("wpi_port_name", "")),
        "coordinates": row.get("wpi_coordinates", ""),
        "locode": clean_locode(row.get("wpi_locode", "")),
        "country_region": row.get("wpi_country_region", ""),
        "fao_areas": ""
    } for _, row in wpi_only.iterrows()
      if clean_port_name(row.get("wpi_port_name", "")) or clean_locode(row.get("wpi_locode", ""))])


def consolidate_ports():
    """
    Compare the prepared WPI and CNSP datasets and generate an analytical
    consolidated overlap table together with matching statistics and audit
    files.

    The consolidated table is not imported directly into Wikibase.
    """
    cnsp, wpi = load_prepared_ports()

    cnsp = prepare_cnsp_for_matching(cnsp)
    wpi = prepare_wpi_for_matching(wpi)

    matched_locode, matched_name, cnsp_only, wpi_only = match_ports(cnsp, wpi)

    out_locode = build_output(matched_locode)
    out_name = build_output(matched_name)
    out_cnsp_only = build_cnsp_only_output(cnsp_only)
    out_wpi_only = build_wpi_only_output(wpi_only)

    final_df = pd.concat(
        [out_locode, out_name, out_cnsp_only, out_wpi_only],
        ignore_index=True
    )

    final_df["port_name"] = final_df["port_name"].apply(clean_port_name)
    final_df["locode"] = final_df["locode"].apply(clean_locode)
    final_df["fao_areas"] = final_df["fao_areas"].apply(format_fao_areas)

    final_df = final_df[
        (final_df["port_name"].astype(str).str.strip() != "") |
        (final_df["locode"].astype(str).str.strip() != "")
    ].copy()

    final_df = final_df.drop_duplicates(
        subset=["port_name", "locode", "coordinates", "country_region", "fao_areas", "sources"]
    ).reset_index(drop=True)

    final_df = final_df.sort_values(by=["port_name", "locode"], na_position="last").reset_index(drop=True)

    print_port_overlap_statistics(
        cnsp=cnsp,
        wpi=wpi,
        matched_locode=matched_locode,
        matched_name=matched_name,
        cnsp_only=cnsp_only,
        wpi_only=wpi_only,
        final_df=final_df
    )

    audit_results = audit_port_overlap(
        cnsp=cnsp,
        wpi=wpi,
        output_prefix=str(AUDIT_OUTPUT_PREFIX)
    )

    final_df.to_csv(CONSOLIDATED_OUTPUT_FILE, sep=";", index=False, encoding="utf-8")

    print(f"File generated: {CONSOLIDATED_OUTPUT_FILE}")
    print(f"Total rows: {len(final_df)}")

# Pipeline execution

In [ ]:
filter_wpi_ports()
filter_cnsp_ports()
consolidate_ports()

WPI file successfully generated
WPI exported rows: 1201
WPI rows with valid coordinates: 1201
WPI rows without valid coordinates: 0
CNSP file generated: C:\Users\profesor\Documents\EFFKG\source_data\processed\ports\cnsp_ports_ue_filtered.csv
CNSP resulting rows: 10066

PORT OVERLAP STATISTICS
------------------------------
cnsp_source_records: 10066
wpi_source_records: 1201
matched_by_locode: 599
matched_by_normalized_name: 9
total_cross_source_matches: 608
locode_matches_pct: 98.52
normalized_name_matches_pct: 1.48
unmatched_cnsp_records: 9460
unmatched_wpi_records: 593
final_consolidated_rows: 10661

PORTS MATCHED BY NORMALIZED NAME
--------------------------------
     matching_method  _cnsp_row_id  _wpi_row_id cnsp_port_name  wpi_port_name cnsp_name_clean cnsp_locode wpi_locode cnsp_country    wpi_country cnsp_region                 wpi_region               cnsp_coordinates               wpi_coordinates
normalized_port_name          2503          505       Neustadt       Neustadt  